# ChakraXAttnUNet — Kaggle dataset audit and GPU pilot

Run the cells in order. Enable a GPU in **Notebook options** before the smoke test. The audit must be reviewed before selecting a dataset for training.

In [ ]:
!rm -rf /kaggle/working/chakramodel
!git clone -q --branch gokzz-glitch-lightweight-transformer-unet https://github.com/Gokzz-glitch/chakramodel.git /kaggle/working/chakramodel
!python /kaggle/working/chakramodel/data/scripts/audit_kaggle_datasets.py --root /kaggle/input --output /kaggle/working/dataset_audit

In [ ]:
from pathlib import Path
import json

report = json.loads(Path('/kaggle/working/dataset_audit/summary.json').read_text())
for item in report:
    print(item['dataset'])
    print(' images:', item['image_count'], 'videos:', item['video_count'], 'mask candidates:', item['mask_candidate_count'])
    print(' folders:', item['top_folders'][:5])
    print(' image samples:', item['sample_images'][:3])
    print(' mask samples:', item['sample_masks'][:3])
    print()

## Select a confirmed labeled source

Edit the next cell only after inspecting the audit output. `IMAGE_ROOT` and `MASK_ROOT` must contain real segmentation images with matching stems. Do not use a raw video-only dataset here.

In [ ]:
from pathlib import Path

# Example only. Replace these after reviewing summary.json.
IMAGE_ROOT = Path('/kaggle/input/REPLACE_OWNER/REPLACE_DATASET/images')
MASK_ROOT = Path('/kaggle/input/REPLACE_OWNER/REPLACE_DATASET/masks')
assert IMAGE_ROOT.is_dir(), f'Missing image directory: {IMAGE_ROOT}'
assert MASK_ROOT.is_dir(), f'Missing mask directory: {MASK_ROOT}'
print(IMAGE_ROOT, MASK_ROOT)

In [ ]:
!python /kaggle/working/chakramodel/data/scripts/prepare_segmentation_manifest.py \
  --images "$IMAGE_ROOT" \
  --masks "$MASK_ROOT" \
  --output /kaggle/working/prepared \
  --materialize

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU in Notebook options before training.')
print(torch.cuda.get_device_name(0))

In [ ]:
!python /kaggle/working/chakramodel/src/chakra_transformer/train_xattn_unet.py \
  --images /kaggle/working/prepared/images/train \
  --masks /kaggle/working/prepared/masks/train \
  --val-images /kaggle/working/prepared/images/val \
  --val-masks /kaggle/working/prepared/masks/val \
  --output /kaggle/working/xattn_smoke \
  --epochs 3 --batch-size 4 --size 352